# U2T02: SimCSE — Train Your Own Sentence Embedding Model

Este notebook contiene la implementación completa para replicar y evaluar **SimCSE** (*Gao et al., 2021*):
1. **Verificación de baselines:** `bert-base-uncased` (59.31 dev / 47.29 test) y `SBERT-2019` (80.77 dev / 76.98 test).
2. **Carga y validación de datos:** `snli_train_100k.jsonl` (165,528 oraciones únicas y 33,351 pares supervisados).
3. **Entrenamiento:** SimCSE Unsupervised (§3) y Supervised con *hard negatives* (§4).
4. **Ablaciones:** Misma máscara de dropout (unsupervised) y Hard negatives OFF (supervised).
5. **Métricas de geometría:** Alignment y Uniformity (Wang & Isola, 2020).
6. **Exportación y publicación:** Formato `sentence-transformers` en Hugging Face Hub con verificación de recarga.

### Paso 0: Configuración en Google Colab o Entorno Local
Si estás corriendo este notebook en **Google Colab**, la siguiente celda se encarga de clonar tu repositorio o configurar la carpeta `src/` automáticamente en el entorno virtual de Colab.

In [ ]:
# 1. Instalar dependencias necesarias
!pip install -q torch transformers datasets sentence-transformers scipy scikit-learn matplotlib seaborn huggingface_hub accelerate

In [ ]:
import os
import sys
from pathlib import Path

# Localizar la raíz del proyecto o configurar la ruta en Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("💻 Ejecutando en Google Colab.")
    # Si clonaste el repositorio en Colab, descomenta y pon tu repo:
    # !git clone https://github.com/TU_USUARIO/U2T02-SimCSE.git
    # %cd U2T02-SimCSE

# Añadir la raíz al path para importar src
for p in [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/U2T2"), Path("/content/U2T02-SimCSE")]:
    if (p / "src").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        os.chdir(p)
        print(f"Directorio raíz establecido en: {p}")
        break

# Si src no existe (porque solo se subió el archivo .ipynb a Colab), lo creamos directamente:
if not os.path.exists("src"):
    print("⚠️ No se encontró la carpeta 'src/'. Creando módulos esenciales directamente en el entorno...")
    os.makedirs("src", exist_ok=True)
    
    # 1. data_loader.py
    with open("src/data_loader.py", "w", encoding="utf-8") as f:
        f.write('''import json, os, torch\nfrom typing import Dict, List, Optional, Tuple\nfrom torch.utils.data import Dataset\nfrom datasets import load_dataset\n\nclass UnsupervisedSimCSEDataset(Dataset):\n    def __init__(self, sentences: List[str]): self.sentences = sentences\n    def __len__(self): return len(self.sentences)\n    def __getitem__(self, idx: int): return self.sentences[idx]\n\nclass SupervisedSimCSEDataset(Dataset):\n    def __init__(self, triplets: List[Dict], use_hard_negatives: bool = True):\n        self.triplets = triplets; self.use_hard_negatives = use_hard_negatives\n    def __len__(self): return len(self.triplets)\n    def __getitem__(self, idx: int):\n        it = self.triplets[idx]\n        return {"premise": it["premise"], "entailment": it["entailment"], "contradiction": it.get("contradiction") if self.use_hard_negatives else None}\n\ndef load_snli_100k(file_path: str):\n    if not os.path.exists(file_path):\n        raise FileNotFoundError(f"No se encontro {file_path}. Sube el archivo a la barra lateral de Colab.")\n    records = []\n    with open(file_path, "r", encoding="utf-8") as f:\n        for line in f:\n            line = line.strip()\n            if line: records.append(json.loads(line))\n    unique = set()\n    for r in records:\n        p, h = r.get("premise", "").strip(), r.get("hypothesis", "").strip()\n        if p: unique.add(p)\n        if h: unique.add(h)\n    unsupervised_sentences = sorted(list(unique))\n    premise_map = {}\n    for r in records:\n        lbl, p, h = r.get("label"), r.get("premise", "").strip(), r.get("hypothesis", "").strip()\n        if not p or not h or lbl not in (0, 1, 2): continue\n        if p not in premise_map: premise_map[p] = {"entailment": [], "contradiction": []}\n        if lbl == 0: premise_map[p]["entailment"].append(h)\n        elif lbl == 2: premise_map[p]["contradiction"].append(h)\n    supervised = []\n    for p, hyps in premise_map.items():\n        ents, contras = hyps["entailment"], hyps["contradiction"]\n        for i, ent in enumerate(ents):\n            contra = contras[i % len(contras)] if contras else None\n            supervised.append({"premise": p, "entailment": ent, "contradiction": contra})\n    return unsupervised_sentences, supervised\n\ndef load_stsb_data():\n    ds = load_dataset("sentence-transformers/stsb")\n    dev = [{"sentence1": r["sentence1"], "sentence2": r["sentence2"], "score": float(r["score"])} for r in ds["validation"]]\n    test = [{"sentence1": r["sentence1"], "sentence2": r["sentence2"], "score": float(r["score"])} for r in ds["test"]]\n    return dev, test\n''')

    # 2. evaluate.py
    with open("src/evaluate.py", "w", encoding="utf-8") as f:
        f.write('''import numpy as np, torch, torch.nn.functional as F\nfrom scipy.stats import spearmanr\nfrom sentence_transformers import SentenceTransformer\n\ndef compute_alignment(x: torch.Tensor, y: torch.Tensor, alpha: float = 2.0):\n    return (torch.norm(x - y, p=2, dim=1) ** alpha).mean().item()\n\ndef compute_uniformity(x: torch.Tensor, t: float = 2.0):\n    sq_pdist = torch.pdist(x, p=2) ** 2\n    return torch.log(torch.mean(torch.exp(-t * sq_pdist))).item()\n\ndef encode_sentences(sentences, model, tokenizer, batch_size=64, device="cpu", pooling="cls"):\n    all_embs = []\n    model.eval()\n    with torch.no_grad():\n        for i in range(0, len(sentences), batch_size):\n            batch = sentences[i:i + batch_size]\n            inputs = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)\n            if hasattr(model, "get_pooled_embedding"):\n                embs = model.get_pooled_embedding(inputs["input_ids"], inputs["attention_mask"], inputs.get("token_type_ids"), return_for_eval=True)\n            else:\n                out = model(**inputs).last_hidden_state\n                if pooling == "cls": embs = out[:, 0]\n                elif pooling == "mean":\n                    mask = inputs["attention_mask"].unsqueeze(-1).expand(out.size()).float()\n                    embs = (out * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)\n            all_embs.append(F.normalize(embs, p=2, dim=1).cpu())\n    return torch.cat(all_embs, dim=0)\n\ndef evaluate_stsb(model, tokenizer, stsb_data, batch_size=64, device="cpu", pooling="cls", compute_geometry=True):\n    s1 = [d["sentence1"] for d in stsb_data]; s2 = [d["sentence2"] for d in stsb_data]; scores = np.array([d["score"] for d in stsb_data])\n    emb1 = encode_sentences(s1, model, tokenizer, batch_size, device, pooling)\n    emb2 = encode_sentences(s2, model, tokenizer, batch_size, device, pooling)\n    cos_sims = (emb1 * emb2).sum(dim=1).numpy()\n    spearman, _ = spearmanr(cos_sims, scores)\n    res = {"spearman": float(spearman * 100.0), "cosine_similarities": cos_sims, "ground_truth_scores": scores}\n    if compute_geometry:\n        pos = scores >= 4.0\n        res["alignment"] = compute_alignment(emb1[pos], emb2[pos]) if pos.sum() > 0 else compute_alignment(emb1, emb2)\n        res["uniformity"] = compute_uniformity(torch.cat([emb1, emb2], dim=0))\n    return res\n\ndef evaluate_sentence_transformer(model_name, stsb_data):\n    m = SentenceTransformer(model_name)\n    s1 = [d["sentence1"] for d in stsb_data]; s2 = [d["sentence2"] for d in stsb_data]; scores = np.array([d["score"] for d in stsb_data])\n    e1 = torch.tensor(m.encode(s1, normalize_embeddings=True)); e2 = torch.tensor(m.encode(s2, normalize_embeddings=True))\n    spearman, _ = spearmanr((e1 * e2).sum(dim=1).numpy(), scores)\n    pos = scores >= 4.0\n    return {"spearman": float(spearman * 100.0), "alignment": compute_alignment(e1[pos], e2[pos]) if pos.sum() > 0 else compute_alignment(e1, e2), "uniformity": compute_uniformity(torch.cat([e1, e2], dim=0))}\n''')

    # 3. models.py
    with open("src/models.py", "w", encoding="utf-8") as f:
        f.write('''import torch, torch.nn as nn, torch.nn.functional as F\nfrom transformers import AutoModel, AutoConfig\n\nclass MLPPooler(nn.Module):\n    def __init__(self, hidden_size): super().__init__(); self.dense = nn.Linear(hidden_size, hidden_size); self.act = nn.Tanh()\n    def forward(self, x): return self.act(self.dense(x))\n\nclass SimCSE(nn.Module):\n    def __init__(self, model_name_or_path="bert-base-uncased", pooling="cls", temperature=0.05, dropout_rate=0.1, use_mlp=True):\n        super().__init__()\n        self.config = AutoConfig.from_pretrained(model_name_or_path)\n        self.config.attention_probs_dropout_prob = dropout_rate; self.config.hidden_dropout_prob = dropout_rate\n        self.encoder = AutoModel.from_pretrained(model_name_or_path, config=self.config)\n        self.pooling = pooling; self.temperature = temperature; self.use_mlp = use_mlp\n        self.mlp = MLPPooler(self.config.hidden_size) if self.use_mlp else nn.Identity()\n    def get_pooled_embedding(self, input_ids, attention_mask, token_type_ids=None, return_for_eval=False):\n        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}\n        if token_type_ids is not None: kwargs["token_type_ids"] = token_type_ids\n        out = self.encoder(**kwargs).last_hidden_state\n        if self.pooling == "cls": rep = out[:, 0]\n        elif self.pooling == "mean":\n            m = attention_mask.unsqueeze(-1).expand(out.size()).float(); rep = (out * m).sum(1) / torch.clamp(m.sum(1), min=1e-9)\n        if return_for_eval or not self.use_mlp: return rep\n        return self.mlp(rep)\n    def forward_unsupervised(self, batch_inputs, same_dropout_ablation=False):\n        ids, mask, tt = batch_inputs["input_ids"], batch_inputs["attention_mask"], batch_inputs.get("token_type_ids")\n        z1 = self.get_pooled_embedding(ids, mask, tt, return_for_eval=False)\n        z2 = z1 if same_dropout_ablation else self.get_pooled_embedding(ids, mask, tt, return_for_eval=False)\n        z1_n, z2_n = F.normalize(z1, p=2, dim=1), F.normalize(z2, p=2, dim=1)\n        sim = torch.mm(z1_n, z2_n.transpose(0, 1)) / self.temperature\n        labels = torch.arange(z1_n.size(0), device=z1_n.device)\n        return F.cross_entropy(sim, labels), sim\n    def forward_supervised(self, p_in, ent_in, contra_in=None, use_hard_negatives=True):\n        hp = F.normalize(self.get_pooled_embedding(p_in["input_ids"], p_in["attention_mask"], p_in.get("token_type_ids")), p=2, dim=1)\n        he = F.normalize(self.get_pooled_embedding(ent_in["input_ids"], ent_in["attention_mask"], ent_in.get("token_type_ids")), p=2, dim=1)\n        sim_pos = torch.mm(hp, he.transpose(0, 1)) / self.temperature\n        if use_hard_negatives and contra_in is not None:\n            hc = F.normalize(self.get_pooled_embedding(contra_in["input_ids"], contra_in["attention_mask"], contra_in.get("token_type_ids")), p=2, dim=1)\n            sim_mat = torch.cat([sim_pos, torch.mm(hp, hc.transpose(0, 1)) / self.temperature], dim=1)\n        else: sim_mat = sim_pos\n        labels = torch.arange(hp.size(0), device=hp.device)\n        return F.cross_entropy(sim_mat, labels), sim_mat\n''')
    print("Módulos creados exitosamente en 'src/'.")

import torch
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo de cómputo activo: {device}")

## 1. Sanity Check de Baselines en STS-B
Validamos que el código de evaluación replique con exactitud los valores de referencia del PDF:
- **Raw `bert-base-uncased` (mean pooling):** 59.31 Dev / 47.29 Test
- **SBERT-2019 (`bert-base-nli-mean-tokens`):** 80.77 Dev / 76.98 Test

In [ ]:
from src.data_loader import load_stsb_data
from src.evaluate import evaluate_stsb, evaluate_sentence_transformer
from transformers import AutoModel, AutoTokenizer

dev_stsb, test_stsb = load_stsb_data()

# 1. Raw bert-base-uncased con mean pooling
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
raw_bert = AutoModel.from_pretrained("bert-base-uncased").to(device)

raw_dev = evaluate_stsb(raw_bert, tokenizer, dev_stsb, device=device, pooling="mean")
raw_test = evaluate_stsb(raw_bert, tokenizer, test_stsb, device=device, pooling="mean")
print(f"Raw BERT (mean): Dev={raw_dev['spearman']:.2f} (Ref: 59.31) | Test={raw_test['spearman']:.2f} (Ref: 47.29)")

# 2. SBERT-2019 (bert-base-nli-mean-tokens)
sbert_dev = evaluate_sentence_transformer("bert-base-nli-mean-tokens", dev_stsb)
sbert_test = evaluate_sentence_transformer("bert-base-nli-mean-tokens", test_stsb)
print(f"SBERT-2019:     Dev={sbert_dev['spearman']:.2f} (Ref: 80.77) | Test={sbert_test['spearman']:.2f} (Ref: 76.98)")

## 2. Validación de Datos: SNLI 100k
Verificamos que `snli_train_100k.jsonl` produzca exactamente 165,528 oraciones únicas y 33,351 pares supervisados.

In [ ]:
from src.data_loader import load_snli_100k

data_path = "snli_train_100k.jsonl"
if os.path.exists(data_path):
    unsupervised_sentences, supervised_items = load_snli_100k(data_path)
    print(f"Oraciones únicas (Unsupervised): {len(unsupervised_sentences)}")
    print(f"Pares supervisados: {len(supervised_items)}")
else:
    print(f"⚠️ Por favor sube '{data_path}' a la carpeta actual (panel izquierdo en Colab) antes de entrenar.")

## 3. Entrenamiento Unsupervised SimCSE (Eq. 1)
Entrenamos con aumento de datos basado en dropout independiente.

In [ ]:
!python -m src.train --mode unsupervised --data_path snli_train_100k.jsonl --batch_size 64 --lr 3e-5 --epochs 1 --seed 42

## 4. Ablación Unsupervised: Mismo Dropout Mask
Comprobamos el colapso de representación al usar la misma máscara de dropout ($z_1 = z_2$).

In [ ]:
!python -m src.train --mode unsupervised --data_path snli_train_100k.jsonl --same_dropout_ablation --batch_size 64 --lr 3e-5 --epochs 1 --seed 42

## 5. Entrenamiento Supervised SimCSE con Hard Negatives (Eq. 5)
Entrenamos con pares premisa-implicación y contradicciones como hard negatives.

In [ ]:
!python -m src.train --mode supervised --data_path snli_train_100k.jsonl --batch_size 64 --lr 5e-5 --epochs 3 --seed 42

## 6. Ablación Supervised: Hard Negatives OFF
Comprobamos el impacto de remover las contradicciones del denominador de la pérdida.

In [ ]:
!python -m src.train --mode supervised --data_path snli_train_100k.jsonl --no_hard_negatives --batch_size 64 --lr 5e-5 --epochs 3 --seed 42

## 7. Exportación y Publicación en Hugging Face Hub (Parte 7)
Exportamos el mejor modelo a formato `sentence-transformers`, lo subimos al Hub y verificamos la igualdad numérica recargándolo desde el Hub.

In [ ]:
# Ajusta tu usuario y token de Hugging Face
HF_USERNAME = "TU_USUARIO"
HF_TOKEN = "TU_HF_TOKEN"
BEST_CKPT = "runs/simcse_supervised_seed42/best_checkpoint"

!python -m src.export_and_publish \
    --checkpoint_dir {BEST_CKPT} \
    --export_dir ./st_best_model \
    --mode supervised \
    --push_to_hub \
    --repo_id f"{HF_USERNAME}/simcse-bert-base-snli" \
    --token {HF_TOKEN}